In [1]:
"""Fermi-gated Bayesian neural network experiment framework.

This module provides a reproducible experimental scaffold for evaluating a
Fermi-gated ensemble of Bayesian neural network approximations against common
baselines.

The implementation is intentionally linear and explicit. Each major step is
spelled out in code blocks with comments so the workflow is easy to audit and
adapt.

Notes
-----
- The code uses Monte Carlo dropout as a practical Bayesian approximation.
- The gating rule is energy-based and can be swapped for a true variational BNN
  or any probabilistic learner that exposes predictive NLL / ELBO.
- The experiment suite is designed for tabular and small-to-medium benchmark
  datasets. For image benchmarks, replace the dataset loader and model factory.

Examples
--------
Run a classification benchmark:

>>> python fermi_gated_bnn_experiments.py --task classification --dataset breast_cancer

Run a regression benchmark:

>>> python fermi_gated_bnn_experiments.py --task regression --dataset diabetes
"""

from __future__ import annotations

import argparse
import json
import math
import os
import random
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_breast_cancer, load_diabetes, load_digits, load_wine
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


# =========================
# Step 1: Configuration
# =========================


@dataclass
class ExperimentConfig:
    """Configuration for the experiment suite.

    Attributes
    ----------
    task : str
        Either ``classification`` or ``regression``.
    dataset : str
        Dataset name. Supported values are ``breast_cancer``, ``wine``,
        ``digits`` for classification and ``diabetes`` for regression.
    seeds : list[int]
        Random seeds for repeated runs.
    num_models : int
        Number of Bayesian network replicas in the ensemble.
    hidden_dim : int
        Width of hidden layers.
    num_layers : int
        Number of hidden layers.
    dropout_rate : float
        Dropout probability used for approximate Bayesian inference.
    lr : float
        Learning rate.
    batch_size : int
        Mini-batch size.
    epochs : int
        Number of training epochs per model.
    mc_samples : int
        Number of stochastic forward passes for evaluation.
    beta : float
        Inverse temperature in the Fermi gate.
    mu_strategy : str
        Strategy for the Fermi threshold. Use ``median`` or ``learned``.
    top_k : int
        Hard cap on the number of active models during sparse evaluation.
    lambda_complexity : float
        Penalty weight for model complexity.
    lambda_kl : float
        Penalty weight for KL regularization.
    lambda_causal : float
        Penalty weight for causal violation penalty.
    output_dir : str
        Directory for all experiment artifacts.
    test_size : float
        Fraction of data reserved for the final test split.
    val_size : float
        Fraction of the remaining training data reserved for validation.
    """

    task: str = "classification"
    dataset: str = "breast_cancer"
    seeds: List[int] = field(default_factory=lambda: [7, 13, 21])
    num_models: int = 8
    hidden_dim: int = 64
    num_layers: int = 2
    dropout_rate: float = 0.2
    lr: float = 1e-3
    batch_size: int = 64
    epochs: int = 50
    mc_samples: int = 30
    beta: float = 4.0
    mu_strategy: str = "median"
    top_k: int = 3
    lambda_complexity: float = 1e-4
    lambda_kl: float = 1e-4
    lambda_causal: float = 0.0
    output_dir: str = "outputs_fermi_bnn"
    test_size: float = 0.2
    val_size: float = 0.2


# =========================
# Step 2: Reproducibility
# =========================


def set_seed(seed: int) -> None:
    """Set the random seed for Python, NumPy, and PyTorch.

    Parameters
    ----------
    seed : int
        Seed value used across libraries.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =========================
# Step 3: Dataset loading
# =========================


def load_tabular_dataset(name: str, task: str) -> Tuple[np.ndarray, np.ndarray]:
    """Load a supported benchmark dataset.

    Parameters
    ----------
    name : str
        Dataset name.
    task : str
        ``classification`` or ``regression``.

    Returns
    -------
    X : ndarray of shape (n_samples, n_features)
        Feature matrix.
    y : ndarray of shape (n_samples,)
        Target vector.
    """

    if task == "classification":
        if name == "breast_cancer":
            data = load_breast_cancer()
        elif name == "wine":
            data = load_wine()
        elif name == "digits":
            data = load_digits()
        else:
            raise ValueError(f"Unsupported classification dataset: {name}")
    elif task == "regression":
        if name == "diabetes":
            data = load_diabetes()
        else:
            raise ValueError(f"Unsupported regression dataset: {name}")
    else:
        raise ValueError(f"Unsupported task: {task}")

    X = np.asarray(data.data, dtype=np.float32)
    y = np.asarray(data.target)
    return X, y


def prepare_splits(
    X: np.ndarray,
    y: np.ndarray,
    task: str,
    test_size: float,
    val_size: float,
    seed: int,
) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """Create train, validation, and test splits with standard scaling.

    Parameters
    ----------
    X : ndarray
        Input features.
    y : ndarray
        Targets.
    task : str
        Task type.
    test_size : float
        Fraction reserved for test.
    val_size : float
        Fraction reserved for validation from the remaining data.
    seed : int
        Random seed.

    Returns
    -------
    splits : dict
        Dictionary with keys ``train``, ``val``, and ``test``.
    """

    stratify = y if task == "classification" else None
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=stratify,
    )
    stratify_train = y_train_val if task == "classification" else None
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=val_size,
        random_state=seed,
        stratify=stratify_train,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    return {
        "train": (X_train, y_train),
        "val": (X_val, y_val),
        "test": (X_test, y_test),
    }


# =========================
# Step 4: Bayesian model approximation
# =========================


class BayesianMLP(nn.Module):
    """Monte Carlo dropout network used as a Bayesian approximation.

    Parameters
    ----------
    input_dim : int
        Input dimensionality.
    output_dim : int
        Number of classes for classification or 1 for regression.
    hidden_dim : int
        Hidden layer width.
    num_layers : int
        Number of hidden layers.
    dropout_rate : float
        Dropout probability.
    task : str
        ``classification`` or ``regression``.
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_dim: int,
        num_layers: int,
        dropout_rate: float,
        task: str,
    ) -> None:
        super().__init__()
        self.task = task
        layers: List[nn.Module] = []
        current_dim = input_dim
        for _ in range(num_layers):
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute raw network outputs."""

        return self.network(x)

    def predictive_distribution(self, x: torch.Tensor, mc_samples: int) -> torch.Tensor:
        """Return MC predictive samples.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
        mc_samples : int
            Number of stochastic forward passes.

        Returns
        -------
        torch.Tensor
            Stacked predictive samples of shape ``(mc_samples, batch, output_dim)``.
        """

        self.train()
        samples = []
        for _ in range(mc_samples):
            samples.append(self.forward(x))
        return torch.stack(samples, dim=0)


# =========================
# Step 5: Losses and metrics
# =========================


def classification_nll(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Cross-entropy negative log-likelihood."""

    return F.cross_entropy(logits, targets.long())


def regression_nll(predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Gaussian NLL proxy using mean squared error."""

    return F.mse_loss(predictions.squeeze(-1), targets.float())


def complexity_penalty(model: nn.Module) -> torch.Tensor:
    """Simple parameter-count complexity penalty."""

    total_params = sum(param.numel() for param in model.parameters())
    device = next(model.parameters()).device
    return torch.tensor(float(total_params), dtype=torch.float32, device=device)


def kl_proxy(model: nn.Module) -> torch.Tensor:
    """Proxy KL term for Monte Carlo dropout models.

    This is a placeholder regularizer. A true variational BNN can replace this
    function with an analytic KL divergence term.
    """

    penalty = torch.tensor(0.0, device=next(model.parameters()).device)
    for param in model.parameters():
        penalty = penalty + torch.sum(param.pow(2))
    return penalty


def causal_violation_proxy(x: torch.Tensor, predictions: torch.Tensor) -> torch.Tensor:
    """Placeholder causal violation penalty.

    This function is deliberately explicit so the user can replace it with a
    domain-specific do-calculus or invariance penalty.
    """

    del x
    return predictions.abs().mean() * 0.0


def expected_calibration_error(probabilities: np.ndarray, targets: np.ndarray, n_bins: int = 10) -> float:
    """Compute the expected calibration error for classification.

    Parameters
    ----------
    probabilities : ndarray
        Class probabilities of shape ``(n_samples, n_classes)``.
    targets : ndarray
        Ground-truth class labels.
    n_bins : int
        Number of confidence bins.

    Returns
    -------
    float
        Expected calibration error.
    """

    confidences = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    accuracies = (predictions == targets).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for start, end in zip(bins[:-1], bins[1:]):
        mask = (confidences > start) & (confidences <= end)
        if np.any(mask):
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            ece += np.abs(bin_acc - bin_conf) * mask.mean()
    return float(ece)


# =========================
# Step 6: Training and evaluation
# =========================


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    """Create a PyTorch data loader."""

    tensor_x = torch.from_numpy(X).float()
    if y.dtype.kind in {"i", "u"}:
        tensor_y = torch.from_numpy(y).long()
    else:
        tensor_y = torch.from_numpy(y).float()
    dataset = TensorDataset(tensor_x, tensor_y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def train_single_model(
    model: BayesianMLP,
    train_loader: DataLoader,
    val_loader: DataLoader,
    task: str,
    lr: float,
    epochs: int,
    lambda_kl: float,
) -> Dict[str, float]:
    """Train a single Bayesian approximation network.

    Parameters
    ----------
    model : BayesianMLP
        Model to train.
    train_loader : DataLoader
        Training data loader.
    val_loader : DataLoader
        Validation data loader.
    task : str
        Task type.
    lr : float
        Learning rate.
    epochs : int
        Number of epochs.
    lambda_kl : float
        Weight of the KL proxy.

    Returns
    -------
    dict
        Training summary including final validation NLL.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            if task == "classification":
                nll = classification_nll(outputs, batch_y)
            else:
                nll = regression_nll(outputs, batch_y)
            kl = kl_proxy(model)
            loss = nll + lambda_kl * kl / max(1.0, float(sum(p.numel() for p in model.parameters())))
            loss.backward()
            optimizer.step()

    val_nll = evaluate_nll(model, val_loader, task, mc_samples=10)
    return {"val_nll": float(val_nll)}


def evaluate_nll(model: BayesianMLP, loader: DataLoader, task: str, mc_samples: int) -> float:
    """Evaluate predictive negative log-likelihood proxy."""

    device = next(model.parameters()).device
    model.eval()
    total = 0.0
    count = 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            samples = []
            for _ in range(mc_samples):
                samples.append(model(batch_x))
            stacked = torch.stack(samples, dim=0)
            mean_pred = stacked.mean(dim=0)
            if task == "classification":
                nll = classification_nll(mean_pred, batch_y)
            else:
                nll = regression_nll(mean_pred, batch_y)
            total += float(nll.item()) * batch_x.shape[0]
            count += batch_x.shape[0]
    return total / max(1, count)


def collect_mc_predictions(
    model: BayesianMLP,
    X: np.ndarray,
    mc_samples: int,
) -> np.ndarray:
    """Collect Monte Carlo predictions from one model."""

    device = next(model.parameters()).device
    tensor_x = torch.from_numpy(X).float().to(device)
    samples = model.predictive_distribution(tensor_x, mc_samples=mc_samples)
    return samples.detach().cpu().numpy()


# =========================
# Step 7: Fermi gating
# =========================


def compute_model_energy(
    model: BayesianMLP,
    val_loader: DataLoader,
    task: str,
    lambda_complexity: float,
    lambda_causal: float,
    mc_samples: int,
) -> float:
    """Compute the energy score for a trained model.

    The default energy is the validation NLL plus complexity and causal
    penalties. Lower values indicate better models.
    """

    device = next(model.parameters()).device
    val_nll = evaluate_nll(model, val_loader, task=task, mc_samples=mc_samples)
    comp = float(complexity_penalty(model).detach().cpu().item())
    comp = comp / max(1.0, float(sum(p.numel() for p in model.parameters())))

    causal = 0.0
    model.eval()
    with torch.no_grad():
        for batch_x, _ in val_loader:
            batch_x = batch_x.to(device)
            pred_samples = model.predictive_distribution(batch_x, mc_samples=5)
            causal_tensor = causal_violation_proxy(batch_x, pred_samples)
            causal += float(causal_tensor.detach().cpu().item()) * batch_x.shape[0]
    causal = causal / max(1, len(val_loader.dataset))

    energy = val_nll + lambda_complexity * comp + lambda_causal * causal
    return float(energy)


def fermi_gate(energies: Sequence[float], beta: float, mu_strategy: str, top_k: int) -> np.ndarray:
    """Convert model energies into Fermi occupancy scores.

    Parameters
    ----------
    energies : sequence of float
        Energy values for the trained models.
    beta : float
        Inverse temperature.
    mu_strategy : str
        Threshold strategy. Use ``median`` or ``learned``.
    top_k : int
        Optional hard cap on active models.

    Returns
    -------
    ndarray
        Gate values in ``[0, 1]``.
    """

    energies = np.asarray(energies, dtype=np.float64)
    if mu_strategy == "median":
        mu = float(np.median(energies))
    elif mu_strategy == "learned":
        mu = float(np.mean(energies))
    else:
        raise ValueError(f"Unsupported mu_strategy: {mu_strategy}")

    scores = 1.0 / (1.0 + np.exp(beta * (energies - mu)))
    if top_k > 0 and top_k < len(scores):
        keep = np.argsort(scores)[::-1][:top_k]
        sparse_scores = np.zeros_like(scores)
        sparse_scores[keep] = scores[keep]
        total = sparse_scores.sum()
        if total > 0:
            sparse_scores = sparse_scores / total
        return sparse_scores

    total = scores.sum()
    if total > 0:
        scores = scores / total
    return scores


def softmax_weights(energies: Sequence[float]) -> np.ndarray:
    """Softmax baseline over negative energies."""

    energies = np.asarray(energies, dtype=np.float64)
    logits = -energies
    logits = logits - logits.max()
    weights = np.exp(logits)
    return weights / weights.sum()


def inverse_nll_weights(energies: Sequence[float]) -> np.ndarray:
    """Simple inverse-energy baseline."""

    energies = np.asarray(energies, dtype=np.float64)
    inv = 1.0 / (energies + 1e-8)
    return inv / inv.sum()


# =========================
# Step 8: Ensemble prediction
# =========================


def ensemble_predict_classification(
    models: Sequence[BayesianMLP],
    weights: np.ndarray,
    X: np.ndarray,
    mc_samples: int,
) -> np.ndarray:
    """Predict class probabilities with weighted model aggregation."""

    model_probs = []
    for model in models:
        samples = collect_mc_predictions(model, X, mc_samples=mc_samples)
        mean_logits = samples.mean(axis=0)
        probs = torch.softmax(torch.from_numpy(mean_logits), dim=-1).numpy()
        model_probs.append(probs)
    stacked = np.stack(model_probs, axis=0)
    return np.tensordot(weights, stacked, axes=(0, 0))


def ensemble_predict_regression(
    models: Sequence[BayesianMLP],
    weights: np.ndarray,
    X: np.ndarray,
    mc_samples: int,
) -> np.ndarray:
    """Predict regression outputs with weighted model aggregation."""

    model_preds = []
    for model in models:
        samples = collect_mc_predictions(model, X, mc_samples=mc_samples)
        mean_pred = samples.mean(axis=0).squeeze(-1)
        model_preds.append(mean_pred)
    stacked = np.stack(model_preds, axis=0)
    return np.tensordot(weights, stacked, axes=(0, 0))


# =========================
# Step 9: Baselines and metrics
# =========================


def classification_metrics(y_true: np.ndarray, probs: np.ndarray) -> Dict[str, float]:
    """Compute classification metrics."""

    pred = probs.argmax(axis=1)
    metrics = {"accuracy": float(accuracy_score(y_true, pred)), "ece": float(expected_calibration_error(probs, y_true))}
    if probs.shape[1] == 2:
        metrics["auroc"] = float(roc_auc_score(y_true, probs[:, 1]))
    return metrics


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute regression metrics."""

    return {
        "rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


# =========================
# Step 10: Experimental runner
# =========================


def build_model(input_dim: int, output_dim: int, cfg: ExperimentConfig, task: str) -> BayesianMLP:
    """Create one Bayesian approximation network."""

    return BayesianMLP(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dim=cfg.hidden_dim,
        num_layers=cfg.num_layers,
        dropout_rate=cfg.dropout_rate,
        task=task,
    )


def run_single_seed(cfg: ExperimentConfig, seed: int) -> Dict[str, float]:
    """Run the complete experimental pipeline for a single seed."""

    set_seed(seed)
    X, y = load_tabular_dataset(cfg.dataset, cfg.task)
    splits = prepare_splits(X, y, cfg.task, cfg.test_size, cfg.val_size, seed)
    X_train, y_train = splits["train"]
    X_val, y_val = splits["val"]
    X_test, y_test = splits["test"]

    train_loader = make_loader(X_train, y_train, cfg.batch_size, shuffle=True)
    val_loader = make_loader(X_val, y_val, cfg.batch_size, shuffle=False)
    test_loader = make_loader(X_test, y_test, cfg.batch_size, shuffle=False)

    if cfg.task == "classification":
        output_dim = int(np.unique(y).shape[0])
    else:
        output_dim = 1

    input_dim = X.shape[1]
    models: List[BayesianMLP] = []
    energies: List[float] = []

    # Step 10.1: Train candidate Bayesian networks.
    for model_idx in range(cfg.num_models):
        model_seed = seed + 1000 * (model_idx + 1)
        set_seed(model_seed)
        model = build_model(input_dim, output_dim, cfg, cfg.task)
        train_single_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            task=cfg.task,
            lr=cfg.lr,
            epochs=cfg.epochs,
            lambda_kl=cfg.lambda_kl,
        )
        energy = compute_model_energy(
            model=model,
            val_loader=val_loader,
            task=cfg.task,
            lambda_complexity=cfg.lambda_complexity,
            lambda_causal=cfg.lambda_causal,
            mc_samples=cfg.mc_samples,
        )
        models.append(model)
        energies.append(energy)

    # Step 10.2: Compute the Fermi gate and baseline weights.
    fermi_weights = fermi_gate(energies, beta=cfg.beta, mu_strategy=cfg.mu_strategy, top_k=cfg.top_k)
    softmax_w = softmax_weights(energies)
    inv_w = inverse_nll_weights(energies)
    uniform_w = np.ones(len(models), dtype=np.float64) / float(len(models))

    # Step 10.3: Evaluate the proposed method and baselines.
    results: Dict[str, float] = {}

    if cfg.task == "classification":
        y_test_int = y_test.astype(int)
        fermi_probs = ensemble_predict_classification(models, fermi_weights, X_test, cfg.mc_samples)
        softmax_probs = ensemble_predict_classification(models, softmax_w, X_test, cfg.mc_samples)
        inv_probs = ensemble_predict_classification(models, inv_w, X_test, cfg.mc_samples)
        uniform_probs = ensemble_predict_classification(models, uniform_w, X_test, cfg.mc_samples)

        fermi_metrics = classification_metrics(y_test_int, fermi_probs)
        soft_metrics = classification_metrics(y_test_int, softmax_probs)
        inv_metrics = classification_metrics(y_test_int, inv_probs)
        uni_metrics = classification_metrics(y_test_int, uniform_probs)

        for key, value in fermi_metrics.items():
            results[f"fermi_{key}"] = value
        for key, value in soft_metrics.items():
            results[f"softmax_{key}"] = value
        for key, value in inv_metrics.items():
            results[f"inverse_{key}"] = value
        for key, value in uni_metrics.items():
            results[f"uniform_{key}"] = value

    else:
        fermi_pred = ensemble_predict_regression(models, fermi_weights, X_test, cfg.mc_samples)
        softmax_pred = ensemble_predict_regression(models, softmax_w, X_test, cfg.mc_samples)
        inv_pred = ensemble_predict_regression(models, inv_w, X_test, cfg.mc_samples)
        uniform_pred = ensemble_predict_regression(models, uniform_w, X_test, cfg.mc_samples)

        fermi_metrics = regression_metrics(y_test, fermi_pred)
        soft_metrics = regression_metrics(y_test, softmax_pred)
        inv_metrics = regression_metrics(y_test, inv_pred)
        uni_metrics = regression_metrics(y_test, uniform_pred)

        for key, value in fermi_metrics.items():
            results[f"fermi_{key}"] = value
        for key, value in soft_metrics.items():
            results[f"softmax_{key}"] = value
        for key, value in inv_metrics.items():
            results[f"inverse_{key}"] = value
        for key, value in uni_metrics.items():
            results[f"uniform_{key}"] = value

    results["seed"] = seed
    results["mean_energy"] = float(np.mean(energies))
    results["std_energy"] = float(np.std(energies))
    results["fermi_active_models"] = float(np.sum(fermi_weights > 0.0))
    results["fermi_entropy"] = float(-np.sum(np.where(fermi_weights > 0, fermi_weights * np.log(fermi_weights + 1e-12), 0.0)))
    return results


def run_experiments(cfg: ExperimentConfig) -> pd.DataFrame:
    """Run repeated experiments across all configured seeds."""

    os.makedirs(cfg.output_dir, exist_ok=True)
    rows = []
    for seed in cfg.seeds:
        row = run_single_seed(cfg, seed)
        rows.append(row)
    df = pd.DataFrame(rows)
    summary_path = Path(cfg.output_dir) / f"results_{cfg.task}_{cfg.dataset}.csv"
    df.to_csv(summary_path, index=False)

    summary = df.mean(numeric_only=True).to_dict()
    summary_path_json = Path(cfg.output_dir) / f"summary_{cfg.task}_{cfg.dataset}.json"
    with open(summary_path_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    return df


# =========================
# Step 11: CLI
# =========================


def parse_args() -> ExperimentConfig:
    """Parse command-line arguments into an ExperimentConfig object."""

    parser = argparse.ArgumentParser(description="Fermi-gated Bayesian neural network experiments")
    parser.add_argument("--task", type=str, default="classification")
    parser.add_argument("--dataset", type=str, default="breast_cancer")
    parser.add_argument("--seeds", type=int, nargs="*", default=[7, 13, 21])
    parser.add_argument("--num_models", type=int, default=8)
    parser.add_argument("--hidden_dim", type=int, default=64)
    parser.add_argument("--num_layers", type=int, default=2)
    parser.add_argument("--dropout_rate", type=float, default=0.2)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--mc_samples", type=int, default=30)
    parser.add_argument("--beta", type=float, default=4.0)
    parser.add_argument("--mu_strategy", type=str, default="median")
    parser.add_argument("--top_k", type=int, default=3)
    parser.add_argument("--lambda_complexity", type=float, default=1e-4)
    parser.add_argument("--lambda_kl", type=float, default=1e-4)
    parser.add_argument("--lambda_causal", type=float, default=0.0)
    parser.add_argument("--output_dir", type=str, default="outputs_fermi_bnn")
    parser.add_argument("--test_size", type=float, default=0.2)
    parser.add_argument("--val_size", type=float, default=0.2)
    args = parser.parse_args()

    return ExperimentConfig(
        task=args.task,
        dataset=args.dataset,
        seeds=args.seeds,
        num_models=args.num_models,
        hidden_dim=args.hidden_dim,
        num_layers=args.num_layers,
        dropout_rate=args.dropout_rate,
        lr=args.lr,
        batch_size=args.batch_size,
        epochs=args.epochs,
        mc_samples=args.mc_samples,
        beta=args.beta,
        mu_strategy=args.mu_strategy,
        top_k=args.top_k,
        lambda_complexity=args.lambda_complexity,
        lambda_kl=args.lambda_kl,
        lambda_causal=args.lambda_causal,
        output_dir=args.output_dir,
        test_size=args.test_size,
        val_size=args.val_size,
    )


def main() -> None:
    """Entry point for the experiment runner."""

    cfg = parse_args()
    os.makedirs(cfg.output_dir, exist_ok=True)
    cfg_path = Path(cfg.output_dir) / f"config_{cfg.task}_{cfg.dataset}.json"
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)
    df = run_experiments(cfg)
    print(df)
    print("\nMean metrics:")
    print(df.mean(numeric_only=True))


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] [--task TASK] [--dataset DATASET]
                                [--seeds [SEEDS ...]]
                                [--num_models NUM_MODELS]
                                [--hidden_dim HIDDEN_DIM]
                                [--num_layers NUM_LAYERS]
                                [--dropout_rate DROPOUT_RATE] [--lr LR]
                                [--batch_size BATCH_SIZE] [--epochs EPOCHS]
                                [--mc_samples MC_SAMPLES] [--beta BETA]
                                [--mu_strategy MU_STRATEGY] [--top_k TOP_K]
                                [--lambda_complexity LAMBDA_COMPLEXITY]
                                [--lambda_kl LAMBDA_KL]
                                [--lambda_causal LAMBDA_CAUSAL]
                                [--output_dir OUTPUT_DIR]
                                [--test_size TEST_SIZE] [--val_size VAL_SIZE]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/shar

SystemExit: 2

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
